In [1]:
# ============================================================================
# STEP 1: SECURE API KEY SETUP - No .env file needed
# ============================================================================
import os
import getpass
import tempfile
import atexit
from pathlib import Path
from collections import Counter
import json
import re
from typing import List, Dict, Any, Tuple, Optional
from datetime import datetime
# Global variables for secure key storage (in memory only)
_SECURE_KEYS = {}
_TEMP_ENV_FILE = None

def secure_get_key(service_name, key_name, prompt_message=None):
    """Get API key securely - stored only in memory"""
    global _SECURE_KEYS
    
    # Return from memory if already entered
    if key_name in _SECURE_KEYS:
        return _SECURE_KEYS[key_name]
    
    # Prompt user for key
    if prompt_message is None:
        prompt_message = f"Enter your {service_name} API key: "
    
    print(f"\n🔐 {service_name} API Key Required")
    print("-" * 50)
    key = getpass.getpass(prompt_message)
    
    if not key:
        raise ValueError(f"No API key provided for {service_name}")
    
    # Store in memory only (no disk write)
    _SECURE_KEYS[key_name] = key
    
    print(f"✓ {service_name} API key loaded (memory only - will be lost on kernel restart)")
    return key

def secure_load_to_env():
    """Load secure keys into os.environ for compatibility with existing code"""
    for key_name, key_value in _SECURE_KEYS.items():
        os.environ[key_name] = key_value
    print("✓ Keys loaded to environment variables")

def secure_cleanup():
    """Clear keys from memory (call when done)"""
    global _SECURE_KEYS
    _SECURE_KEYS.clear()
    # Also clear from os.environ
    for key in list(os.environ.keys()):
        if 'API_KEY' in key or 'GEMINI' in key or 'DEEPSEEK' in key:
            os.environ.pop(key, None)
    print("✓ API keys cleared from memory")

# Register cleanup on kernel shutdown (optional)
atexit.register(secure_cleanup)

print("✓ Secure API key manager ready")
print("  • Keys stored in memory only")
print("  • No .env file created")
print("  • Keys lost when kernel restarts")

✓ Secure API key manager ready
  • Keys stored in memory only
  • No .env file created
  • Keys lost when kernel restarts


In [2]:
# ============================================================================
# OPTIONAL: DEEPSEEK API SETUP (SECURE VERSION)
# ============================================================================

from openai import OpenAI

# Get DeepSeek API key securely
DEEPSEEK_API_KEY = secure_get_key("DeepSeek", "DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = input("\nDeepSeek base URL (press Enter for default): ").strip()
if not DEEPSEEK_BASE_URL:
    DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# Initialize DeepSeek client

print(f"✓ DeepSeek client configured")
print(f"  Base URL: {DEEPSEEK_BASE_URL}")

# Optional: Wrapper function
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)



🔐 DeepSeek API Key Required
--------------------------------------------------
✓ DeepSeek API key loaded (memory only - will be lost on kernel restart)
✓ DeepSeek client configured
  Base URL: https://api.deepseek.com


In [10]:
def get_cot_prompt(numbers: List[int]) -> str:
    """
    Single-path Chain-of-Thought prompt for 24-game (4 numbers, target 24).
    Forces linear reduction without backtracking.
    """
    numbers_str = ", ".join(map(str, numbers))
    
    prompt_old = f"""Numbers: [{numbers_str}] Target: 24

Combine the 4 numbers using exactly 3 steps. Each step picks two numbers, 
applies one operation (+, -, *, /), and replaces them with the result.

Strict format — one path only, no alternatives, no backtracking:

Step 1: [A] op [B] = [R1]. Remaining: [remaining 3 numbers including R1]
Step 2: [C] op [D] = [R2]. Remaining: [remaining 2 numbers including R2]  
Step 3: [E] op [F] = [final]

Final result: [final]
ANSWER: SUCCESS if final == 24, otherwise FAIL

Solve for [{numbers_str}]:"""
    prompt = f"""Numbers: [{numbers_str}] Target: 24

Combine the 4 numbers using exactly 3 steps. Each step picks two numbers,
applies one operation (+, -, *, /), and replaces them with the result.

ONE attempt only. Do not retry if you fail.

Step 1: [A] op [B] = [R1]. Remaining: [remaining 3 numbers including R1]
Step 2: [C] op [D] = [R2]. Remaining: [remaining 2 numbers including R2]
Step 3: [E] op [F] = [final]

Final result: [final]
ANSWER: SUCCESS if final == 24, otherwise FAIL

Solve for [{numbers_str}]:"""
    
    return prompt


def extract_cot_answer(response_text: str) -> Tuple[Optional[bool], Optional[str]]:
    """
    Extract SUCCESS/FAIL and reasoning steps from CoT response.
    Returns: (success_flag, reasoning_summary)
    """
    answer_match = re.search(r'ANSWER:\s*(SUCCESS|FAIL)', response_text, re.IGNORECASE)
    success = None
    if answer_match:
        success = answer_match.group(1).upper() == "SUCCESS"
    
    # Extract step-by-step reasoning
    reasoning = ""
    if "ANSWER:" in response_text:
        reasoning = response_text[:response_text.index("ANSWER:")].strip()
    else:
        reasoning = response_text.strip()
    
    return success, reasoning

In [16]:
def run_cot_benchmark(
    numbers: List[int],
    k: int = 1,
    temperature: float = 0.0,
    max_tokens: int = 120,
    model: str = "deepseek-chat"
) -> Dict[str, Any]:
    """
    Run Chain-of-Thought benchmark for 24-game.
    
    Args:
        numbers: List of 4 integers
        k: Number of independent runs for self-consistency
        temperature: Sampling temperature (0 for deterministic, >0 for diversity)
        max_tokens: Maximum tokens per response
        model: Model to use (default "deepseek-chat")
    
    Returns:
        Dictionary with benchmark results
    """
    
    results = {
        "numbers": numbers,
        "target": 24,
        "k": k,
        "temperature": temperature,
        "model": model,
        "timestamp": datetime.now().isoformat(),
        "individual_runs": [],
        "cot_sc": {
            "success_count": 0,
            "success_rate": 0.0,
            "majority_answer": None,
            "answer_distribution": {},
            "total_api_calls": k,
            "total_tokens_used": 0,
            "confidence": 0.0,
            "majority_consistency": 0.0
        }
    }
    
    prompt = get_cot_prompt(numbers)
    
    for run_id in range(k):
        print(f"  Run {run_id + 1}/{k}...", end=" ")
        
        try:
            if k==1:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a precise mathematical reasoner. Follow instructions exactly. Do not backtrack or try alternative paths."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=temperature if k > 1 else 0,
                    max_tokens=max_tokens
                )
            elif k>1:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a mathematical reasoner. Solve the problem using the exact format provided."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=temperature if k > 1 else 0,
                    max_tokens=max_tokens
                )

            response_text = response.choices[0].message.content
            usage = response.usage
            
            success, reasoning = extract_cot_answer(response_text)
            
            # Extract the 3-step reduction sequence
            steps = []
            step_pattern = r'Step \d+:(.*?)(?=Step \d+|ANSWER:|$)'
            step_matches = re.findall(step_pattern, response_text, re.DOTALL)
            steps = [s.strip() for s in step_matches]
            
            run_data = {
                "run_id": run_id + 1,
                "success": success,
                "reasoning": reasoning,
                "steps": steps,
                "full_response": response_text,
                "tokens": {
                    "prompt": usage.prompt_tokens,
                    "completion": usage.completion_tokens,
                    "total": usage.total_tokens
                },
                "temperature_used": temperature if k > 1 else 0
            }
            
            results["individual_runs"].append(run_data)
            results["cot_sc"]["total_tokens_used"] += usage.total_tokens
            
            if success:
                results["cot_sc"]["success_count"] += 1
                print(f"✓ SUCCESS")
            else:
                print(f"✗ FAIL")
                
        except Exception as e:
            print(f"❌ Error: {e}")
            results["individual_runs"].append({
                "run_id": run_id + 1,
                "success": None,
                "error": str(e),
                "reasoning": None,
                "steps": [],
                "full_response": None,
                "tokens": {"prompt": 0, "completion": 0, "total": 0}
            })
    
    # Compute CoT-SC statistics
    total_runs = len(results["individual_runs"])
    successful_runs = [r for r in results["individual_runs"] if r.get("success") is True]
    results["cot_sc"]["success_count"] = len(successful_runs)
    results["cot_sc"]["success_rate"] = len(successful_runs) / total_runs if total_runs > 0 else 0.0
    
    # Get majority answer
    answers = [r["success"] for r in results["individual_runs"] if r.get("success") is not None]
    if answers:
        answer_counts = Counter(answers)
        results["cot_sc"]["answer_distribution"] = {str(k): v for k, v in answer_counts.items()}
        majority_answer = answer_counts.most_common(1)[0][0]
        results["cot_sc"]["majority_answer"] = "SUCCESS" if majority_answer else "FAIL"
        results["cot_sc"]["majority_consistency"] = max(answer_counts.values()) / len(answers)
    
    results["cot_sc"]["confidence"] = results["cot_sc"]["success_rate"]
    
    return results

In [15]:
def save_results(results: Dict[str, Any], output_dir: str = "cot_benchmark_results"):
    """Save benchmark results to JSON file (one file per problem)."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Create filename from numbers
    numbers_str = "_".join(map(str, results["numbers"]))
    filename = f"cot_{numbers_str}_target{results['target']}_k{results['k']}.json"
    filepath = os.path.join(output_dir, filename)
    
    # Convert non-serializable types
    results_copy = results.copy()
    results_copy["timestamp"] = str(results_copy["timestamp"])
    
    with open(filepath, 'w') as f:
        json.dump(results_copy, f, indent=2)
    
    print(f"\n✓ Results saved to: {filepath}")
    return filepath


def load_results(numbers: List[int], target: int = 24, k: int = None, output_dir: str = "cot_benchmark_results") -> Optional[Dict]:
    """Load previously saved results for a problem."""
    numbers_str = "_".join(map(str, numbers))
    pattern = f"cot_{numbers_str}_target{target}"
    
    if k:
        filename = f"{pattern}_k{k}.json"
        filepath = os.path.join(output_dir, filename)
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                return json.load(f)
    else:
        # Find latest file matching pattern
        import glob
        files = glob.glob(os.path.join(output_dir, f"{pattern}_k*.json"))
        if files:
            latest = max(files, key=os.path.getctime)
            with open(latest, 'r') as f:
                return json.load(f)
    
    return None

In [17]:
def print_summary(results: Dict[str, Any]):
    """Print clean summary for 24-game results."""
    print("\n" + "="*60)
    print(f"📊 24-Game CoT Benchmark Summary")
    print("="*60)
    print(f"Numbers: {results['numbers']} → Target: 24")
    print(f"Model: {results['model']}")
    print(f"K (runs): {results['k']}")
    print(f"Temperature: {results['temperature']}")
    print("-"*60)
    
    sc = results['cot_sc']
    print(f"✅ SUCCESS rate: {sc['success_rate']*100:.1f}% ({sc['success_count']}/{results['k']})")
    print(f"🎯 Majority answer: {sc['majority_answer']}")
    print(f"📊 Answer distribution: {sc['answer_distribution']}")
    print(f"🔒 Majority consistency: {sc['majority_consistency']*100:.1f}%")
    print(f"📝 Total tokens used: {sc['total_tokens_used']}")
    print("-"*60)
    
    for run in results['individual_runs']:
        status = "✅ SUCCESS" if run.get('success') else "❌ FAIL" if run.get('success') is False else "⚠️ ERROR"
        tokens = run.get('tokens', {}).get('total', 0)
        steps_count = len(run.get('steps', []))
        print(f"  Run {run['run_id']}: {status} (steps: {steps_count}, tokens: {tokens})")
    
    print("="*60)


# Example 1: Single-path CoT (k=1, deterministic)
# print("\n" + "="*60)
# print("Example 1: Single-path CoT (k=1, temperature=0)")
# print("="*60)

# results_single = run_cot_benchmark(
#     numbers=[4, 4, 4, 4],
#     k=1,
#     temperature=0.0
# )
# print_summary(results_single)
# save_results(results_single)


# # Example 2: CoT-SC (k=5, diverse paths)
# print("\n" + "="*60)
# print("Example 2: CoT-Self-Consistency (k=5, temperature=0.7)")
# print("="*60)

# results_sc = run_cot_benchmark(
#     numbers=[4, 4, 4, 4],
#     k=5,
#     temperature=0.7
# )
# print_summary(results_sc)
# save_results(results_sc)


# Example 3: Classic hard problem
print("\n" + "="*60)
print("Example 3: Hard problem [3, 3, 8, 8]")
print("="*60)

results_hard = run_cot_benchmark(
    numbers=[1,5, 9, 13],
    k=5,
    temperature=0.8
)
print_summary(results_hard)
save_results(results_hard)


Example 3: Hard problem [3, 3, 8, 8]
  Run 1/5... ✗ FAIL
  Run 2/5... ✗ FAIL
  Run 3/5... ✗ FAIL
  Run 4/5... ✗ FAIL
  Run 5/5... ✗ FAIL

📊 24-Game CoT Benchmark Summary
Numbers: [1, 5, 9, 13] → Target: 24
Model: deepseek-chat
K (runs): 5
Temperature: 0.8
------------------------------------------------------------
✅ SUCCESS rate: 0.0% (0/5)
🎯 Majority answer: FAIL
📊 Answer distribution: {'False': 5}
🔒 Majority consistency: 100.0%
📝 Total tokens used: 1335
------------------------------------------------------------
  Run 1: ❌ FAIL (steps: 3, tokens: 267)
  Run 2: ❌ FAIL (steps: 3, tokens: 267)
  Run 3: ❌ FAIL (steps: 3, tokens: 267)
  Run 4: ❌ FAIL (steps: 3, tokens: 267)
  Run 5: ❌ FAIL (steps: 3, tokens: 267)

✓ Results saved to: cot_benchmark_results\cot_1_5_9_13_target24_k5.json


'cot_benchmark_results\\cot_1_5_9_13_target24_k5.json'